# K03_03 – Modellvalidierung und Kreuzvalidierung – Studentenversion

Diese Fassung ist für die **aktive Mitarbeit im Kurs** gedacht.

In diesem Notebook geht es um eine zentrale methodische Frage:

> Wie schätzen wir möglichst fair ein, wie gut ein Modell auf neuen Daten funktioniert?

## Lernziele
Nach diesem Notebook können Sie:
- erklären, warum Bewertung auf Trainingsdaten problematisch ist
- `train_test_split` korrekt einsetzen
- den Grundgedanken der Kreuzvalidierung verstehen
- Mittelwert und Streuung von CV-Scores interpretieren

## 1. Ein schlechter Start: Training und Bewertung auf denselben Daten

Wir verwenden den Iris-Datensatz und ein sehr einfaches Modell.


In [ ]:
from sklearn.datasets import load_iris
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
import numpy as np

iris = load_iris()
X = iris.data
y = iris.target

model = KNeighborsClassifier(n_neighbors=1)
model.fit(X, y)
y_pred_train = model.predict(X)

acc_train = accuracy_score(y, y_pred_train)
print(f"Accuracy auf denselben Trainingsdaten: {acc_train:.3f}")


### Beobachtung
Dieser Wert wirkt sehr gut.  
Aber er ist **zu optimistisch**, weil wir das Modell auf denselben Daten bewerten, auf denen es gelernt hat.


## Mini-Übung 1
Warum ist diese Methode keine faire Schätzung für neue, unbekannte Daten?


## 2. Bessere Methode: Train/Test-Split


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

model = KNeighborsClassifier(n_neighbors=1)
model.fit(X_train, y_train)
y_pred_test = model.predict(X_test)

acc_test = accuracy_score(y_test, y_pred_test)
print(f"Accuracy auf Testdaten: {acc_test:.3f}")


### Merksatz
Die Testdaten sollen das Modell **erst ganz am Ende** sehen.


## 3. Kreuzvalidierung

Ein einzelner Train/Test-Split kann zufällig günstig oder ungünstig sein.  
Deshalb verwenden wir oft **Cross-Validation**.


In [ ]:
cv_scores = cross_val_score(
    KNeighborsClassifier(n_neighbors=5),
    X, y,
    cv=5,
    scoring="accuracy"
)

print("Einzelne CV-Scores:", np.round(cv_scores, 3))
print("Mittelwert:", round(cv_scores.mean(), 3))
print("Standardabweichung:", round(cv_scores.std(), 3))


## Mini-Übung 2
1. Warum unterscheiden sich die einzelnen Folds?
2. Was sagt der Mittelwert aus?
3. Was sagt die Standardabweichung aus?


## 4. Vergleich mehrerer Werte für k


In [ ]:
results = []

for k in range(1, 16):
    scores = cross_val_score(
        KNeighborsClassifier(n_neighbors=k),
        X, y,
        cv=5,
        scoring="accuracy"
    )
    results.append((k, scores.mean(), scores.std()))

import pandas as pd
df_results = pd.DataFrame(results, columns=["k", "mean_accuracy", "std_accuracy"])
df_results.round(3)


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(df_results["k"], df_results["mean_accuracy"], marker="o")
plt.xlabel("k")
plt.ylabel("Mittlere CV-Accuracy")
plt.title("Kreuzvalidierung für verschiedene k-Werte")
plt.show()


### Beobachtung
Hier sehen wir, dass Modellleistung vom Hyperparameter `k` abhängt.  
Kreuzvalidierung hilft uns, diese Wahl robuster zu beurteilen.


## 5. StratifiedKFold

Bei Klassifikation ist es oft sinnvoll, die Klassenverteilung in den Folds möglichst ähnlich zu halten.


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores_strat = cross_val_score(
    KNeighborsClassifier(n_neighbors=5),
    X, y,
    cv=cv,
    scoring="accuracy"
)

print("Scores mit StratifiedKFold:", np.round(scores_strat, 3))
print("Mittelwert:", round(scores_strat.mean(), 3))


## Mini-Übung 3
Warum ist bei Klassifikationsproblemen eine ähnliche Klassenverteilung in allen Folds oft sinnvoll?


## 6. Zusammenfassung

- Bewertung auf Trainingsdaten ist methodisch ungeeignet
- ein Holdout-Test ist besser
- Kreuzvalidierung ist oft robuster
- wir betrachten dabei nicht nur einen Score, sondern auch seine Streuung
